In [ ]:
# Example 1: Inspect the package's public API surface.
import gwlearn

public_names = [n for n in dir(gwlearn) if not n.startswith('_')]
print(f"Public names on gwlearn ({len(public_names)}):")
for name in public_names:
    obj = getattr(gwlearn, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, "gwlearn should expose at least one public name"

In [ ]:
# Example 2: Build a small synthetic geospatial dataset for downstream cells.
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

rng = np.random.default_rng(seed=42)
n = 60
xs = rng.uniform(0, 10, size=n)
ys = rng.uniform(0, 10, size=n)

# Spatially-varying signal: feature interacts with x-coord.
feat = rng.normal(size=n)
y_cont = 0.5 * feat * xs + rng.normal(scale=0.1, size=n)
y_bin = (y_cont > np.median(y_cont)).astype(int)

geom = [Point(x, y) for x, y in zip(xs, ys)]
gdf = gpd.GeoDataFrame({"feat": feat, "y_cont": y_cont, "y_bin": y_bin}, geometry=geom, crs="EPSG:3857")
print(gdf.head())
assert len(gdf) == n

In [ ]:
# Example 3: Instantiate the geographically-weighted classifier.
import gwlearn
from gwlearn.base import BaseClassifier
from sklearn.base import ClassifierMixin
from sklearn.linear_model import LogisticRegression

print(f"GW classifier: {BaseClassifier.__module__}.{BaseClassifier.__qualname__}")
assert issubclass(BaseClassifier, ClassifierMixin), BaseClassifier.__mro__

# It wraps any scikit-learn classifier and fits one local model per location within the bandwidth
clf = BaseClassifier(LogisticRegression, bandwidth=5.0, fixed=True, n_jobs=1)
assert (clf.model, clf.bandwidth, clf.fixed, clf.n_jobs) == (LogisticRegression, 5.0, True, 1), clf.get_params()
print(f"Constructed {type(clf).__name__}(model=LogisticRegression, bandwidth=5.0, fixed=True)")

In [ ]:
# Example 4: Smoke-test a GW model fit/predict if a usable class is exposed.
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.linear_model import LogisticRegression

import gwlearn

rng = np.random.default_rng(seed=7)
n = 40
X = pd.DataFrame({"a": rng.normal(size=n), "b": rng.normal(size=n)})
y = pd.Series((X["a"] + rng.normal(scale=0.2, size=n) > 0).astype(int), name="y")
geom = gpd.GeoSeries([Point(rng.uniform(0, 10), rng.uniform(0, 10)) for _ in range(n)], crs="EPSG:3857")

GWClassifier = getattr(gwlearn, "GWClassifier", None)
if GWClassifier is None:
    print("GWClassifier not exposed at top level; skipping fit/predict.")
else:
    model = GWClassifier(model=LogisticRegression, bandwidth=5.0, fixed=True, n_jobs=1)
    model.fit(X, y, geom)
    print(f"Fitted model: {type(model).__name__}")
    assert hasattr(model, "predict") or hasattr(model, "local_models")

In [ ]:
# Example 5: Verify the package's runtime dependencies are co-importable.
import importlib

expected = ["geopandas", "joblib", "libpysal", "numpy", "pandas", "sklearn", "scipy"]
loaded = {}
for mod in expected:
    m = importlib.import_module(mod)
    loaded[mod] = getattr(m, "__version__", "unknown")

for mod, ver in loaded.items():
    print(f"  {mod}: {ver}")

import gwlearn
assert len(loaded) == len(expected)
assert gwlearn is not None